In [1]:
import os
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from transformers import (
    BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
)
from torch.utils.data import Dataset
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [2]:
# Load data
jailbreak_df = pd.read_csv('/content/all_labelled_cleaned.csv')
regular_df = pd.read_csv('/content/regular_prompts_DAN.csv')
# Preprocessing setup
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

In [8]:
def preprocess_text(text):
    words = text.split()
    return ' '.join([stemmer.stem(word) for word in words if word.lower() not in stop_words])

def create_dataset(df, regular_df, label_name, train=True):
    if train:
        subset_df = df[~df['Labels'].apply(lambda labels: label_name in labels)].copy()
    else:
        subset_df = df[df['Labels'].apply(lambda labels: label_name in labels)].copy()

    subset_df['Jailbreak'] = 1
    subset_df = subset_df[['prompt', 'Jailbreak']]

    sampled_regular = regular_df.sample(n=10*len(subset_df), random_state=1).reset_index(drop=True)
    sampled_regular['Jailbreak'] = 0
    sampled_regular['prompt'] = sampled_regular['prompt'].apply(preprocess_text)
    sampled_regular = sampled_regular[['prompt', 'Jailbreak']]

    combined_df = pd.concat([subset_df, sampled_regular], ignore_index=True)
    return combined_df.sample(frac=1, random_state=1).reset_index(drop=True)


In [9]:
# Create train/test datasets for each label
labels = ['Character Roleplay', 'Superior Model', 'Sudo Mode', 'Simulate Jailbreaking', 'Ethical Appeal']

train_dfs, test_dfs = [], []
for label in labels:
    train_dfs.append(create_dataset(jailbreak_df, regular_df, label, train=True))
    test_dfs.append(create_dataset(jailbreak_df, regular_df, label, train=False))

# Dataset class
class PromptDataset(Dataset):
    def __init__(self, prompts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(prompts, truncation=True, padding=True, max_length=max_length)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
            'labels': torch.tensor(self.labels[idx])
        }

# Compute metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    auc = roc_auc_score(labels, pred.predictions[:,1])
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fnr = fn / (tp + fn) if (tp + fn) > 0 else 0
    return {'accuracy': acc, 'auc': auc, 'f1': f1, 'tpr': tpr, 'fnr': fnr}


In [10]:
# Training loop
os.environ["WANDB_MODE"] = "disabled"
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

metrics_dict = {}

for i, (train_df, test_df) in enumerate(zip(train_dfs, test_dfs)):
    print(labels[i])
    train_dataset = PromptDataset(train_df['prompt'].tolist(), train_df['Jailbreak'].tolist(), tokenizer)
    test_dataset = PromptDataset(test_df['prompt'].tolist(), test_df['Jailbreak'].tolist(), tokenizer)

    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
    model_output_dir = f'./saved_model_{i}'
    os.makedirs(model_output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=model_output_dir,
        num_train_epochs=4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=10,
        weight_decay=0.01,
        logging_dir=os.path.join(model_output_dir, 'logs'),
        logging_steps=10,
        eval_strategy="epoch"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    trainer.save_model(model_output_dir)
    tokenizer.save_pretrained(model_output_dir)

    eval_results = trainer.evaluate()
    metrics_dict[f'Model_{i+1}'] = eval_results

Character Roleplay


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Auc,F1,Tpr,Fnr
1,0.045900,0.051764,0.989876,0.990053,0.946272,0.980682,0.019318
2,0.000400,0.058738,0.989566,0.994178,0.944597,0.978409,0.021591
3,0.000100,0.046941,0.992252,0.994538,0.958077,0.973864,0.026136
4,0.000100,0.036768,0.994318,0.994009,0.968553,0.962500,0.037500


Superior Model


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Auc,F1,Tpr,Fnr
1,0.000100,0.015847,0.997608,0.999985,0.987013,1.000000,0.000000
2,0.000300,0.012402,0.998405,0.999908,0.991150,0.982456,0.017544
3,0.000000,0.000026,1.000000,1.000000,1.000000,1.000000,0.000000
4,0.000000,0.000020,1.000000,1.000000,1.000000,1.000000,0.000000


... (5 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy,Auc,F1,Tpr,Fnr
1,0.000300,0.000258,1.000000,1.000000,1.000000,1.000000,0.000000
2,0.000200,0.041616,0.994652,0.999346,0.969697,0.941176,0.058824
3,0.000200,0.052693,0.992870,0.984121,0.959184,0.921569,0.078431
4,0.000100,0.031304,0.996435,0.986390,0.980000,0.960784,0.039216


Ethical Appeal


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Auc,F1,Tpr,Fnr
1,0.102500,0.064228,0.989091,0.992320,0.936170,0.880000,0.120000
2,0.145200,0.022152,0.992727,0.999680,0.960000,0.960000,0.040000
3,0.083800,0.032694,0.996364,0.999040,0.979592,0.960000,0.040000
4,0.000100,0.060609,0.992727,0.999200,0.958333,0.920000,0.080000
